# Qwen3-VL-Embedding-2B — offline Kaggle full extraction

Attach the three Kaggle Datasets before running, then keep Internet **off**. Their Dataset slugs do not matter: the next cell discovers the required folders automatically.

1. `aichcm-qwen-pilot-code`: this repository's `src/` folder.
2. `aichcm-keyframes-full`: contains one common `keyframes/` directory with every `L##_V###/*.jpg` video to embed.
3. `qwen3-vl-embedding-2b-local`: contains the full local model directory, including `model.safetensors` and `scripts/qwen3_vl_embedding.py`.
4. Optional `aichcm-qwen-offline-wheels`: only needed if Kaggle is missing `qwen-vl-utils` or a sufficiently new Transformers package.
5. Optional prior output Dataset: contains `qwen_full_artifacts_bundle.zip` from an earlier run, allowing the notebook to resume after a session reset.

Complete artifacts are skipped on a rerun. To resume after a session reset, attach the prior output ZIP as item 5. The notebook loads Qwen once, processes all videos at batch size 32, writes only to `/kaggle/working`, and never changes the CLIP pipeline.

In [ ]:
from pathlib import Path
import importlib.util
import os
import re
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')

def discover_exactly_one(description, candidates):
    candidates = list(candidates)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one {description}; found: {candidates}')
    return candidates[0]

def shallow_input_paths(max_depth=6):
    # Never descend into keyframes: 177k JPEGs make recursive glob very slow.
    frontier = [(INPUT_ROOT, 0)]
    while frontier:
        parent, depth = frontier.pop()
        for child in parent.iterdir():
            yield child
            if child.is_dir() and child.name != 'keyframes' and depth < max_depth:
                frontier.append((child, depth + 1))

# Kaggle mounts attached Datasets as /kaggle/input/<dataset-slug>. Discover
# only the shallow Dataset structure, then inspect keyframes one video at a time.
SHALLOW_PATHS = list(shallow_input_paths())
model_weight = discover_exactly_one(
    'Qwen model.safetensors',
    (path for path in SHALLOW_PATHS if path.name == 'model.safetensors'
     and (path.parent / 'scripts' / 'qwen3_vl_embedding.py').is_file()),
)
MODEL_DIR = model_weight.parent
CODE_SRC = discover_exactly_one(
    'Qwen pilot src directory',
    (path for path in SHALLOW_PATHS if path.name == 'src'
     and (path / 'visual_pipeline' / 'qwen3_embedding' / 'extractor.py').is_file()),
)
VIDEO_ID_PATTERN = re.compile(r'L\d+_V\d+')
KEYFRAME_ROOT_CANDIDATES = [
    path for path in SHALLOW_PATHS if path.name == 'keyframes' and path.is_dir()
    and any(child.is_dir() and VIDEO_ID_PATTERN.fullmatch(child.name) for child in path.iterdir())
]
if not KEYFRAME_ROOT_CANDIDATES:
    raise RuntimeError('No full keyframes root found under /kaggle/input')
VIDEO_DIRS_BY_ROOT = {
    root: sorted(
        path for path in root.iterdir()
        if path.is_dir() and VIDEO_ID_PATTERN.fullmatch(path.name)
        and next(path.glob('*.jpg'), None) is not None
    )
    for root in KEYFRAME_ROOT_CANDIDATES
}
largest_count = max(len(paths) for paths in VIDEO_DIRS_BY_ROOT.values())
largest_roots = [root for root, paths in VIDEO_DIRS_BY_ROOT.items() if len(paths) == largest_count]
if len(largest_roots) != 1:
    raise RuntimeError(f'Ambiguous full keyframe roots: {largest_roots}')
KEYFRAME_ROOT = largest_roots[0]
VIDEO_DIRS = VIDEO_DIRS_BY_ROOT[KEYFRAME_ROOT]
IGNORED_KEYFRAME_ROOTS = [root for root in KEYFRAME_ROOT_CANDIDATES if root != KEYFRAME_ROOT]
if not VIDEO_DIRS:
    raise RuntimeError(f'No L##_V### keyframe directories found under {KEYFRAME_ROOT}')
VIDEO_IDS = [path.name for path in VIDEO_DIRS]
qwen_wheel = next((path for path in SHALLOW_PATHS if path.name.startswith('qwen_vl_utils-')
                  and path.suffix == '.whl'), None)
OUTPUT_DIR = Path('/kaggle/working/qwen_full_artifacts')
BATCH_SIZE = 32  # Kaggle RTX 6000 Pro; reduce to 16 only if an OOM occurs.

sys.path.insert(0, str(CODE_SRC))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prior_archives = [path for path in SHALLOW_PATHS if path.name == 'qwen_full_artifacts_bundle.zip']
if len(prior_archives) > 1:
    raise RuntimeError(f'Expected at most one prior artifact ZIP, found: {prior_archives}')
if prior_archives:
    print('Restoring prior artifacts from:', prior_archives[0])
    shutil.unpack_archive(str(prior_archives[0]), str(OUTPUT_DIR))
else:
    print('No prior artifact ZIP attached; starting from empty output.')
if IGNORED_KEYFRAME_ROOTS:
    print('Ignoring smaller keyframe Dataset(s):', IGNORED_KEYFRAME_ROOTS)
print(f'Videos: {len(VIDEO_IDS)}')
print('Keyframe root:', KEYFRAME_ROOT)
print('Model:', MODEL_DIR)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Install the exact attached wheel without resolving its optional video
# dependency. This pilot embeds still images only, so PyAV is not used.
if importlib.util.find_spec('qwen_vl_utils') is None:
    if qwen_wheel is None or not qwen_wheel.is_file():
        raise RuntimeError(
            'qwen-vl-utils wheel is missing under /kaggle/input. '
            'Attach the qwen-wheels Dataset; Internet remains off.'
        )
    print('Installing exact wheel:', qwen_wheel)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index',
        '--no-deps', str(qwen_wheel),
    ], check=True)
import torch
import transformers
import qwen_vl_utils

if tuple(map(int, transformers.__version__.split('.')[:2])) < (4, 57):
    raise RuntimeError(
        f'transformers={transformers.__version__}; Qwen needs >=4.57. Attach a compatible offline wheel Dataset.'
    )
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before running the pilot.')
print('torch:', torch.__version__, 'transformers:', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Offline full extraction. Qwen loads once and completed videos are skipped on rerun.
env = os.environ.copy()
env.update({
    'PYTHONPATH': str(CODE_SRC),
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'QWEN3_VL_EMBED_MODEL': str(MODEL_DIR),
    'QWEN3_VL_EMBED_KEYFRAME_DIR': str(KEYFRAME_ROOT),
    'QWEN3_VL_EMBED_FEATURE_DIR': str(OUTPUT_DIR),
})
command = [
    sys.executable, '-m', 'visual_pipeline.qwen3_embedding.extractor',
    '--all-videos',
    '--continue-on-error',
    '--device', 'cuda',
    '--dtype', 'float16',
    '--batch-size', str(BATCH_SIZE),
]
print('Running:', ' '.join(command))
subprocess.run(command, env=env, check=True)

In [ ]:
# Validate every completed cloud artifact before downloading them.
import json
import numpy as np

validated = []
for manifest_path in sorted(OUTPUT_DIR.glob('L*_V*_manifest.json')):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    video_id = manifest['video_id']
    features = np.load(OUTPUT_DIR / manifest['feature_file'])
    filenames = json.loads((OUTPUT_DIR / manifest['filenames_file']).read_text(encoding='utf-8'))
    assert features.shape == (len(filenames), 2048), (video_id, features.shape)
    assert np.isfinite(features).all(), video_id
    assert np.allclose(np.linalg.norm(features, axis=1), 1.0, atol=1e-3), video_id
    validated.append((video_id, len(filenames)))

completed_ids = {video_id for video_id, _ in validated}
missing_ids = sorted(set(VIDEO_IDS) - completed_ids)
print(f'Validated {len(validated)} videos / {sum(count for _, count in validated)} vectors.')
if missing_ids:
    print('Not yet complete (rerun resumes):', missing_ids)
report_path = OUTPUT_DIR / 'qwen3_extraction_report.json'
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('Failures recorded:', report['failed'])

archive_base = '/kaggle/working/qwen_full_artifacts_bundle'
archive = __import__('shutil').make_archive(archive_base, 'zip', OUTPUT_DIR)
archive_path = Path(archive)
assert archive_path.is_file(), archive_path
print('Created:', archive_path, f'({archive_path.stat().st_size / 1024 / 1024:.1f} MB)')
from IPython.display import FileLink, display
display(FileLink(str(archive_path), result_html_prefix='Download artifacts: '))

## After downloading the ZIP to the local repository

Extract all files into `data/npy_features_qwen3_vl_2b/`, then index only the separate Qwen collection:

```powershell
$env:PYTHONPATH = 'src'
python -m visual_pipeline.qwen3_embedding.database --all-videos --recreate
```

For the first local query test, use the GPU with one text input:

```powershell
$env:PYTHONPATH = 'src'
$env:HF_HUB_OFFLINE = '1'
$env:QWEN3_VL_EMBED_MODEL = (Resolve-Path '.\data\models\Qwen3-VL-Embedding-2B')
& .\.venv-qwen3\Scripts\python.exe -m visual_pipeline.qwen3_embedding.query `
  --video-id L21_V001 --device cuda --dtype float16 --batch-size 1 `
  'mô tả hình ảnh cần tìm'
```

This verifies cloud image vectors and local text vectors share the same Qwen space. It uses only one text input, so it is far lighter than extracting 307 images.